# Quench dynamics of the transverse field Ising model

| | |
|---|---|
| **Level** | Advanced |
| **Time** | 60 to 90 minutes |
| **Prerequisites** | Spin chains, time evolution, Trotterization |
| **Default devices** | IQM Garnet |
| **Hardware jobs** | 8 (one per time point) |
| **Approximate cost** | about 820 credits on Garnet at 500 shots |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*Part of the QUEST notebooks from qBraid: chemistry and physics.*

This notebook simulates the dynamics of a quantum spin chain after a sudden change in its Hamiltonian, called a quench. Quench dynamics is central to current research on thermalization and entanglement growth in isolated quantum systems.

We use the one-dimensional transverse-field Ising model (TFIM). It can be solved exactly by mapping it to free fermions, so quantum simulations can be checked against exact values. Its Trotterized time evolution is also short enough to run on current hardware.

The system starts in the ground state of a Hamiltonian with a large transverse field, which is fully polarized along $x$. It is then quenched to a Hamiltonian with strong Ising interactions and a weak transverse field, and we follow how its magnetization changes over time.

**Learning objectives**

1. Describe quench dynamics as a standard non-equilibrium problem.
2. Build a Trotterized time-evolution circuit for the TFIM.
3. Compare Trotter simulation with exact diagonalization for small systems.
4. See how the Trotter step size trades circuit depth against approximation error.
5. Run the evolution on real hardware and see how noise limits the time that can be reached.
6. Discuss when quantum simulation is expected to outperform classical simulation.

**Background needed:** Hamiltonians, time evolution and expectation values. Spin models are introduced as needed.


## The transverse-field Ising model in one paragraph

Consider a chain of $N$ spin-1/2 particles arranged in 1D. Each spin interacts with its nearest neighbor via an Ising coupling of strength $J$, and each spin feels a uniform transverse magnetic field of strength $h$:

$$\hat{H}_{\text{TFIM}} = -J \sum_{i=1}^{N-1} \hat{Z}_i \hat{Z}_{i+1} - h \sum_{i=1}^{N} \hat{X}_i$$

The two limits are simple. When $h \gg J$, the ground state is the fully-polarized state along the $+x$ direction: $|{+}{+}{+}...{+}\rangle$. When $J \gg h$, the ground state is doubly-degenerate ferromagnetic: $|{\uparrow}{\uparrow}...\rangle$ and $|{\downarrow}{\downarrow}...\rangle$. Somewhere between, at $h/J = 1$ (in the thermodynamic limit), there's a quantum phase transition.

**The quench experiment.** Start in the $h \gg J$ ground state (all spins pointing along $+x$, meaning $\langle X_i \rangle = 1$ and $\langle Z_i \rangle = 0$). Then suddenly change the Hamiltonian to a value that's dominated by Ising interactions. The initial state is not an eigenstate of the new Hamiltonian, so it starts evolving in time. Watch the magnetization $\langle X_i(t) \rangle$ oscillate and decay.

Why bother with such a simple setup? Because this controlled experiment mimics questions from research on thermalization ("does the system reach thermal equilibrium under unitary evolution?"), entanglement dynamics ("how fast does entanglement spread?"), and prethermalization ("do integrable systems relax to non-thermal steady states?").


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import product

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

# System size (small enough for exact diagonalization)
N = 4

# Model parameters for the quench
J_QUENCH = 1.0
H_QUENCH = 0.5   # ratio h/J = 0.5, so we're in the ferromagnetic-dominated regime

# Initial state: |+++...+> (all spins along +x)
# This is the h >> J ground state

print(f"System size N = {N}")
print(f"Post-quench H: J = {J_QUENCH}, h = {H_QUENCH}, h/J = {H_QUENCH/J_QUENCH}")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS = 500
QUEST_JOB_TAGS = {"quest": "chem-quench"}   # labels this notebook's hardware jobs for QUEST usage statistics


## Step 1: build the TFIM Hamiltonian

Construct the Hamiltonian as a sparse Pauli operator on $N$ qubits. Qubit ordering is Qiskit little-endian throughout.


In [ ]:
def build_tfim_hamiltonian(N, J, h):
    """Return TFIM Hamiltonian as SparsePauliOp."""
    pauli_list = []
    # Ising terms: -J * Z_i Z_{i+1}
    for i in range(N - 1):
        label = ['I'] * N
        label[i] = 'Z'
        label[i + 1] = 'Z'
        pauli_list.append((''.join(reversed(label)), -J))
    # Transverse field: -h * X_i
    for i in range(N):
        label = ['I'] * N
        label[i] = 'X'
        pauli_list.append((''.join(reversed(label)), -h))
    return SparsePauliOp.from_list(pauli_list)


H_quench = build_tfim_hamiltonian(N, J_QUENCH, H_QUENCH)
print(f"TFIM Hamiltonian on N={N} qubits, {len(H_quench)} Pauli terms:")
for p, c in zip(H_quench.paulis, H_quench.coeffs):
    print(f"  {p}: {c.real:+.3f}")

## Step 2: exact reference by diagonalization

At $N=4$, the Hilbert space has $2^4 = 16$ dimensions, well within reach of a laptop. Direct diagonalization gives us exact eigenvalues and eigenvectors, which we use to compute the exact time-evolved state $|\psi(t)\rangle = e^{-i \hat{H} t}|\psi(0)\rangle$ and measure exact expectation values.

*Aside on scaling.* For $N=20$, the Hilbert space has a million dimensions. For $N=50$, it's $2^{50} \approx 10^{15}$, larger than any computer can store as a dense state vector. That is the regime where quantum simulation on hardware becomes necessary. But at $N=4$, we're firmly in the "we can double-check with a laptop" regime, and that's what makes this a good pedagogical demo.


In [ ]:
def initial_state_plus(N):
    """Prepare |+++...+> as a Statevector."""
    qc = QuantumCircuit(N)
    for q in range(N):
        qc.h(q)
    return Statevector.from_instruction(qc)


def exact_time_evolve(psi0, hamiltonian, t):
    """Compute exp(-i H t) |psi0> using dense diagonalization."""
    H_matrix = hamiltonian.to_matrix()
    from scipy.linalg import expm
    U = expm(-1j * H_matrix * t)
    return Statevector(U @ psi0.data)


def measure_x_magnetization(state, N):
    """Return <sum_i X_i> / N (average x-magnetization)."""
    total = 0.0
    for i in range(N):
        label = ['I'] * N
        label[i] = 'X'
        op = SparsePauliOp.from_list([(''.join(reversed(label)), 1.0)])
        total += np.real(state.expectation_value(op))
    return total / N


# Initial state and initial magnetization (should be 1.0 for |+...+>)
psi_0 = initial_state_plus(N)
m0 = measure_x_magnetization(psi_0, N)
print(f"Initial <X> per site: {m0:.4f} (expected: 1.000)")

# Sweep time and record exact magnetization
times = np.linspace(0, 4.0, 25)
exact_magnetizations = []
for t in times:
    psi_t = exact_time_evolve(psi_0, H_quench, t)
    exact_magnetizations.append(measure_x_magnetization(psi_t, N))

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(times, exact_magnetizations, 'k-', linewidth=2, label='Exact')
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Time (in units of $1/J$)', fontsize=12)
ax.set_ylabel(r'$\langle X \rangle$ per site', fontsize=12)
ax.set_title(f'Quench dynamics of TFIM (N={N}, J=1, h=0.5): exact reference', fontsize=13, pad=15)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

The exact evolution shows the initial full polarization decaying with oscillations. This is a signature of the system moving away from the highly polarized initial state as the Ising interactions rotate spins away from $+x$. The oscillation period is set by the energy gaps in the post-quench Hamiltonian.

*Physics note: for an infinite chain, this magnetization would decay smoothly to zero at long times, a signature of thermalization in the eigenstate thermalization hypothesis (ETH) sense. At finite $N$, we see revivals and oscillations because the finite spectrum has a finite Heisenberg time. This finite-size effect is a real research topic on its own.*


## Step 3: Trotterized time evolution

The exact matrix exponential $e^{-i\hat{H}t}$ is efficient to compute classically at $N=4$ but not implementable directly on a quantum computer. Instead, we approximate it using the Trotter decomposition:

$$e^{-i\hat{H}t} \approx \left( e^{-i\hat{H}_{ZZ} \Delta t} e^{-i\hat{H}_X \Delta t} \right)^{n_{\text{steps}}}, \quad \Delta t = t / n_{\text{steps}}$$

Each Trotter step alternates between evolving under just the Ising terms (which we can implement with $R_{ZZ}(2 J \Delta t)$ gates on nearest-neighbor pairs) and evolving under just the transverse field (which we can implement with $R_X(2 h \Delta t)$ gates on each qubit).

The Trotter error scales as $O(\Delta t^2)$ per step, so smaller $\Delta t$ (more steps) is more accurate but produces a deeper circuit. This trade-off between accuracy and circuit depth is the central pedagogical point of this section.


In [ ]:
def trotter_step(qc, N, J, h, dt):
    """Apply one first-order Trotter step of TFIM evolution."""
    # Evolve under Ising terms: exp(+i * J * Z_i Z_{i+1} * dt) for each pair
    # In Qiskit, rzz(theta) = exp(-i * theta * ZZ / 2), so we need theta = -2*J*dt for the sign convention
    for i in range(N - 1):
        qc.rzz(-2 * J * dt, i, i + 1)
    # Evolve under transverse field: exp(+i * h * X_i * dt) for each qubit
    # rx(theta) = exp(-i * theta * X / 2), so theta = -2 * h * dt
    for i in range(N):
        qc.rx(-2 * h * dt, i)


def build_trotter_evolution(N, J, h, t, n_steps):
    """Build a circuit that Trotter-evolves the initial |+...+> state for time t."""
    dt = t / n_steps
    qc = QuantumCircuit(N)
    # Prepare |+...+>
    for q in range(N):
        qc.h(q)
    # Apply n_steps Trotter steps
    for _ in range(n_steps):
        trotter_step(qc, N, J, h, dt)
    return qc


# Draw a sample circuit for t=1, n_steps=4
qc_sample = build_trotter_evolution(N, J_QUENCH, H_QUENCH, t=1.0, n_steps=4)
print(f"Circuit depth (t=1, 4 Trotter steps): {qc_sample.depth()}")
print(f"Two-qubit gate count: {qc_sample.count_ops().get('rzz', 0)}")
qc_sample.draw('mpl', fold=100)

## Step 4: Trotter error vs step size (ideal simulator)

Sweep the number of Trotter steps at a fixed final time and see how the approximation converges.


In [ ]:
def simulate_trotter_magnetization(N, J, h, t, n_steps):
    """Return <X> per site from Trotter simulation at time t."""
    qc = build_trotter_evolution(N, J, h, t, n_steps)
    state = Statevector.from_instruction(qc)
    return measure_x_magnetization(state, N)


# For a fixed target time, sweep Trotter steps
t_target = 2.0  # in units of 1/J
n_steps_list = [1, 2, 4, 8, 16, 32]

# Exact reference at this time
exact_at_target = measure_x_magnetization(exact_time_evolve(psi_0, H_quench, t_target), N)

trotter_at_target = [simulate_trotter_magnetization(N, J_QUENCH, H_QUENCH, t_target, ns)
                     for ns in n_steps_list]

errors = [abs(m - exact_at_target) for m in trotter_at_target]

# Plot the convergence
fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(n_steps_list, errors, 'o-', color='#a02580', markersize=10, linewidth=2,
          markeredgecolor='white', markeredgewidth=1.5)
# Expected scaling: error ~ 1/n_steps (for first-order Trotter, error per step is dt^2 = (t/n)^2,
# total accumulated error is n * dt^2 = t^2 / n, so error ~ 1/n_steps)
ax.loglog(n_steps_list, [errors[0] * n_steps_list[0] / n for n in n_steps_list],
          'k--', alpha=0.5, label=r'$\propto 1 / n_{\rm steps}$ scaling')
ax.set_xlabel('Number of Trotter steps', fontsize=12)
ax.set_ylabel(r'$|\langle X \rangle_{\rm Trotter} - \langle X \rangle_{\rm exact}|$', fontsize=12)
ax.set_title(f'Trotter convergence at t = {t_target}', fontsize=13, pad=15)
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"Exact <X> at t={t_target}: {exact_at_target:.4f}")
print(f"Trotter with 32 steps:      {trotter_at_target[-1]:.4f}")

First-order Trotter error scales as $1/n_{\text{steps}}$ for a fixed final time. Doubling the number of steps halves the error, at the cost of doubling the circuit depth.

*Aside:* Higher-order Trotter decompositions (Suzuki-Trotter, second- or fourth-order) achieve better scaling but at the cost of additional structure and more gates per step. For NISQ hardware, first-order Trotter with modest $n_{\text{steps}}$ is often the sweet spot.


## Step 5: Trotter evolution on real hardware

Now the harder experiment. Run the Trotter circuit on real hardware and measure the magnetization at multiple time points. Two effects compete here: Trotter error shrinks as we use more Trotter steps for a given time, while hardware noise grows with circuit depth.

For hardware, we need to be economical: use a modest number of Trotter steps and see how far in time we can trust the results.

For measuring $\langle X_i \rangle$, we need to rotate to the $X$ basis before measuring in the computational basis. This is a Hadamard on each qubit before measurement.


In [ ]:
# Conservative gate set the vendor compilers accept. Verified on Rigetti and IQM.
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']


def build_trotter_with_x_measurement(N, J, h, t, n_steps):
    """Build Trotter evolution followed by Hadamards and computational-basis measurement."""
    qc = build_trotter_evolution(N, J, h, t, n_steps)
    # Rotate to X basis for each qubit
    for q in range(N):
        qc.h(q)
    qc.measure_all()
    return qc


def counts_to_x_magnetization(counts, N):
    """Given measurement counts (in X basis), compute <X> per site."""
    total = sum(counts.values())
    total_mag = 0.0
    for bitstring, count in counts.items():
        # bitstring is in little-endian; bit=0 means <X>=+1 (before H, this is |+>),
        # bit=1 means <X>=-1
        # measure_all() adds a second classical register; some routes return
        # space-separated keys ('1 01'), so strip separators before indexing.
        bits = bitstring.replace(' ', '')
        site_sum = 0.0
        for q in range(N):
            bit = int(bits[-(q + 1)])
            site_sum += (1 - 2 * bit)  # 0 -> +1, 1 -> -1
        total_mag += (site_sum / N) * count / total
    return total_mag


# Set up qBraid device
provider = QbraidProvider()
# Devices are named by qBraid QRN. The README lists devices, prices and availability.
DEVICE_ID = 'aws:iqm:qpu:garnet'
device = provider.get_device(DEVICE_ID)

# Sweep time points on hardware
hw_times = np.linspace(0.25, 3.0, 8)
n_steps_per_time = 4   # small number of Trotter steps

hw_magnetizations = []
for t in hw_times:
    qc = build_trotter_with_x_measurement(N, J_QUENCH, H_QUENCH, t, n_steps_per_time)
    # Decompose to a basis the vendor compilers accept. Rigetti's quilc cannot route
    # rzz ("Requested to rewire RZZ(...), but we don't know how to do this") and nothing
    # in the submission path decomposes it. Verified on Rigetti and IQM; not yet tested
    # on IonQ. Same unitary; the physics is unchanged.
    qc = transpile(qc, basis_gates=HW_BASIS, optimization_level=1)
    depth = qc.depth()
    print(f"t={t:.2f}: circuit depth={depth}, submitting...")

    job = device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS)
    result = job.result()
    counts = result.data.get_counts()
    mag = counts_to_x_magnetization(counts, N)
    hw_magnetizations.append(mag)
    print(f"  <X> = {mag:.4f} (exact: {measure_x_magnetization(exact_time_evolve(psi_0, H_quench, t), N):.4f})")

In [ ]:
# Also compute Trotter on ideal simulator at same n_steps for comparison
ideal_trotter_mags = [simulate_trotter_magnetization(N, J_QUENCH, H_QUENCH, t, n_steps_per_time)
                      for t in hw_times]

# Plot: exact, ideal Trotter, and hardware Trotter
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times, exact_magnetizations, 'k-', linewidth=2, alpha=0.7, label='Exact')
ax.plot(hw_times, ideal_trotter_mags, 's-', color='#2d7a4f', markersize=10, linewidth=2,
        label=f'Trotter, {n_steps_per_time} steps (ideal simulator)',
        markeredgecolor='white', markeredgewidth=1.5)
ax.plot(hw_times, hw_magnetizations, 'o-', color='#c63792', markersize=12, linewidth=2.5,
        label=f'Trotter, {n_steps_per_time} steps (real hardware)',
        markeredgecolor='white', markeredgewidth=1.5)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Time (in units of $1/J$)', fontsize=12)
ax.set_ylabel(r'$\langle X \rangle$ per site', fontsize=12)
ax.set_title('TFIM quench dynamics: exact vs Trotter vs hardware', fontsize=13, pad=15)
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

What you should see: the ideal-simulator Trotter (green squares) tracks the exact curve closely for short times, then deviates as Trotter error accumulates. The hardware curve (pink circles) starts near the ideal but decays much faster, and by later times may saturate near zero regardless of the true value.

**The pedagogical takeaway.** On today's hardware, the accessible evolution time is bounded not by the Trotter approximation but by accumulated gate noise. At short times, hardware agrees with the ideal calculation. At long times, hardware measurements degenerate into random noise, effectively giving $\langle X \rangle \approx 0$ from a fully-mixed state that has no memory of the initial condition. There is a maximum useful evolution time that depends on hardware fidelity and circuit structure. For serious quantum simulation research, extending this maximum time is one of the central problems.

*A comment on quantum advantage.* For this system size ($N=4$), a laptop out-performs any quantum computer that exists today: exact evolution is trivially fast and exact. Quantum advantage for simulation is expected to emerge somewhere around $N=40-50$, where classical state-vector simulation becomes memory-limited, and around $N=80-100$ where even tensor-network approximations struggle. Reaching that regime requires both larger and less noisy hardware than we have now. This notebook is a working demonstration of the technique; the "does it beat classical" question is separate and unresolved.


## When does quantum simulation beat classical?

Three regimes to keep in mind:

1. **Small $N$ (up to ~30 qubits).** Classical simulation is fast and exact. Quantum simulation is educational and interesting but not competitive.
2. **Intermediate $N$ (30-70 qubits).** Classical state-vector methods hit memory limits, but tensor-network methods (matrix product states, PEPS) still work for one-dimensional and quasi-1D systems with limited entanglement. Quantum simulation becomes attractive for higher-dimensional or highly-entangled systems.
3. **Large $N$ (100+ qubits).** Classical methods have fundamental scaling walls. Quantum simulation is the only path forward, provided the hardware is good enough to run circuits deep enough to reach interesting physics.

TFIM in 1D is famously *not* the best case for quantum simulation, because it's exactly solvable via a Jordan-Wigner mapping to free fermions. But it makes an excellent pedagogical example precisely because we have exact reference values against which to benchmark. The interesting quantum simulation targets are 2D and 3D interacting systems (Hubbard model, spin liquids, lattice gauge theories, etc.) where classical methods struggle even at modest sizes.


## Going further
- **Study the quench across the phase transition.** Start in the ground state of $h \gg J$ (paramagnetic) and quench to $h \ll J$ (ferromagnetic), or vice versa. What happens to the magnetization? Does the system show any signature of the underlying phase transition?
- **Add error mitigation.** Apply zero-noise extrapolation (see the [phase estimation notebook](../algorithms/intermediate_02_phase_estimation_precision_vs_noise.ipynb)) to the hardware evolution. How much of the ideal curve can you recover?
- **Higher-order Trotter.** Implement second-order Trotter ($e^{-iH_A dt/2} e^{-iH_B dt} e^{-iH_A dt/2}$) and see how it changes the accuracy vs depth trade-off.
- **Different model.** Replace the transverse-field Ising model with the Heisenberg XXZ model or a random-field Ising model. Each has different physics (thermalization vs many-body localization) that can be probed with quench dynamics.
